# Stokoe — ASL Sign Classifier Training

From-scratch CNN+LSTM. No pretrained weights (Requirement 7).

**Before running:** Upload `stokoe_frames.zip` to your Google Drive root, then run cells in order.

In [ ]:
# ── 1. Mount Drive and unzip data ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import zipfile, pathlib

zip_path = '/content/drive/MyDrive/stokoe_frames.zip'
extract_to = '/content/stokoe'

if not pathlib.Path(extract_to + '/data/manifest.csv').exists():
    print('Extracting...')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_to)
    print('Done.')
else:
    print('Already extracted.')

DATA_DIR = extract_to + '/data'

In [ ]:
# ── 2. Verify GPU and install tensorflowjs for export ────────────────────────
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

!pip install -q tensorflowjs tqdm

In [ ]:
# ── 3. Configuration ─────────────────────────────────────────────────────────
SEQ_LEN   = 16    # frames per clip
FRAME_SIZE = 64   # px
BATCH     = 16
EPOCHS    = 40
LR        = 1e-3
CONFIDENCE_THRESHOLD = 0.72  # documented in pilot (Requirement 8)

In [ ]:
# ── 4. Load and preprocess data ──────────────────────────────────────────────
import csv, pathlib, time
from collections import defaultdict
import numpy as np
from tqdm import tqdm

def load_manifest(data_dir):
    rows = []
    with open(f'{data_dir}/manifest.csv') as f:
        for row in csv.DictReader(f):
            rows.append(row)
    return rows

def load_clip(frames_dir, split, gloss, clip_id, seq_len):
    clip_dir = pathlib.Path(frames_dir) / split / gloss / clip_id
    paths = sorted(clip_dir.glob('frame_*.npy'))
    frames = [np.load(p) for p in paths]
    n = len(frames)
    if n >= seq_len:
        indices = np.linspace(0, n - 1, seq_len, dtype=int)
        sampled = [frames[i] for i in indices]
    else:
        sampled = frames + [frames[-1]] * (seq_len - n)
    return np.stack(sampled, axis=0).astype(np.float32) / 255.0

def build_split(manifest, frames_dir, label_index, split, seq_len, augment=False):
    X, y = [], []
    rows = [r for r in manifest if r['split'] == split]
    for row in tqdm(rows, desc=f'Loading {split}'):
        if row['gloss'] not in label_index:
            continue
        try:
            clip = load_clip(frames_dir, split, row['gloss'], row['clip_id'], seq_len)
        except Exception as e:
            continue
        X.append(clip)
        y.append(label_index[row['gloss']])
        if augment:
            X.append(clip[:, :, ::-1, :])   # horizontal flip
            y.append(label_index[row['gloss']])
            X.append(clip[::-1])             # time reversal
            y.append(label_index[row['gloss']])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

manifest = load_manifest(DATA_DIR)
frames_dir = DATA_DIR + '/frames'

labels = sorted({r['gloss'] for r in manifest if r['split'] == 'train'})
label_index = {l: i for i, l in enumerate(labels)}
NUM_CLASSES = len(labels)
print(f'{NUM_CLASSES} classes')

X_train, y_train = build_split(manifest, frames_dir, label_index, 'train', SEQ_LEN, augment=True)
X_val,   y_val   = build_split(manifest, frames_dir, label_index, 'val',   SEQ_LEN)
X_test,  y_test  = build_split(manifest, frames_dir, label_index, 'test',  SEQ_LEN)

print(f'Train {X_train.shape}  Val {X_val.shape}  Test {X_test.shape}')

In [ ]:
# ── 5. Build model ───────────────────────────────────────────────────────────
from tensorflow import keras
from tensorflow.keras import layers

def cnn_block(x, filters, dropout=0.25):
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(dropout)(x)
    return x

frame_in = keras.Input(shape=(FRAME_SIZE, FRAME_SIZE, 3))
x = cnn_block(frame_in, 32)
x = cnn_block(x, 64)
x = cnn_block(x, 128)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
cnn = keras.Model(frame_in, x, name='frame_cnn')

seq_in = keras.Input(shape=(SEQ_LEN, FRAME_SIZE, FRAME_SIZE, 3), name='sequence')
feats  = layers.TimeDistributed(cnn)(seq_in)
x      = layers.LSTM(128, dropout=0.3, recurrent_dropout=0.2)(feats)
x      = layers.Dense(128, activation='relu')(x)
x      = layers.Dropout(0.4)(x)
out    = layers.Dense(NUM_CLASSES, activation='softmax', name='logits')(x)

model = keras.Model(seq_in, out, name='stokoe_cnn_lstm')
model.summary()

model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

In [ ]:
# ── 6. Train ─────────────────────────────────────────────────────────────────
run_id = time.strftime('%Y%m%d_%H%M%S')
ckpt   = f'/content/stokoe_model_{run_id}.keras'

callbacks = [
    keras.callbacks.ModelCheckpoint(ckpt, monitor='val_accuracy', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=callbacks,
    shuffle=True,
)

In [ ]:
# ── 7. Evaluate ──────────────────────────────────────────────────────────────
loss, acc = model.evaluate(X_test, y_test, batch_size=BATCH, verbose=0)
print(f'Test accuracy: {acc:.4f}  Loss: {loss:.4f}')

preds = model.predict(X_test, batch_size=BATCH, verbose=0).argmax(axis=1)
per_class = defaultdict(lambda: [0, 0])
for true, pred in zip(y_test, preds):
    per_class[labels[true]][1] += 1
    if true == pred:
        per_class[labels[true]][0] += 1

print('\nPer-class accuracy:')
for label in sorted(per_class):
    c, t = per_class[label]
    print(f'  {label:20s}  {c}/{t}  ({100*c//t}%)')

In [ ]:
# ── 8. Export to TensorFlow.js ───────────────────────────────────────────────
import json, subprocess

tfjs_out = f'/content/stokoe_tfjs_{run_id}'

!tensorflowjs_converter --input_format=keras {ckpt} {tfjs_out}

# Write label map and metadata
meta = {
    'labels': {str(i): l for i, l in enumerate(labels)},
    'seq_len': SEQ_LEN,
    'frame_size': FRAME_SIZE,
    'confidence_threshold': CONFIDENCE_THRESHOLD,
    'test_accuracy': round(float(acc), 4),
    'run_id': run_id,
}
pathlib.Path(tfjs_out + '/meta.json').write_text(json.dumps(meta, indent=2))

# Zip for download
zip_out = f'/content/stokoe_model_{run_id}.zip'
!zip -r {zip_out} {tfjs_out}

# Copy to Drive so it persists after session ends
!cp {zip_out} /content/drive/MyDrive/
print(f'Saved to Drive: stokoe_model_{run_id}.zip')

size_mb = sum(pathlib.Path(tfjs_out).stat().st_size for p in pathlib.Path(tfjs_out).rglob('*') if p.is_file()) / 1e6
print(f'Model size: {size_mb:.1f} MB')
print(f'Load in browser: tf.loadLayersModel("/model/model.json")')